### Imports

In [19]:
import json
import os
from pathlib import Path

import requests
import yaml
from dotenv import load_dotenv

load_dotenv()

True

### Load config

In [20]:
CONFIG_PATH = Path("config.yaml")

with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

config

{'github': {'repositories': ['https://github.com/huggingface/transformers',
   'https://github.com/langchain-ai/langchain'],
  'collect': {'issues': True,
   'pull_requests': True,
   'discussions': True,
   'releases': True,
   'contributors': True},
  'limits': {'issues': 100, 'pull_requests': 100}}}

In [21]:
type(config)

dict

### GitHub authentication

In [22]:
token = os.getenv("GITHUB_TOKEN")

if not token:
    raise RuntimeError("GITHUB_TOKEN environment variable is not set.")

headers = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {token}",
    "X-GitHub-Api-Version": "2022-11-28",
}

### parse repository URL

In [8]:
for url in config["github"]["repositories"]:
    print(url)

{'owner': 'huggingface', 'repo': 'transformers'}
{'owner': 'langchain-ai', 'repo': 'langchain'}


In [23]:
def parse_repo_url(url: str) -> tuple[str, str]:
    url = url.rstrip("/")

    parts = url.split("/")

    if len(parts) < 2:
        raise ValueError(f"Invalid GitHub URL: {url}")

    owner = parts[-2]
    repo = parts[-1]

    return owner, repo


repositories = [parse_repo_url(url) for url in config["github"]["repositories"]]

repositories

[('huggingface', 'transformers'), ('langchain-ai', 'langchain')]

### basic GitHub API request

In [24]:
BASE_URL = "https://api.github.com"


def github_get(endpoint: str, params: dict | None = None):
    response = requests.get(
        f"{BASE_URL}{endpoint}",
        headers=headers,
        params=params,
        timeout=30,
    )

    response.raise_for_status()

    return response.json()

In [25]:
owner, repo = repositories[0]

repository = github_get(f"/repos/{owner}/{repo}")

repository

{'id': 155220641,
 'node_id': 'MDEwOlJlcG9zaXRvcnkxNTUyMjA2NDE=',
 'name': 'transformers',
 'full_name': 'huggingface/transformers',
 'private': False,
 'owner': {'login': 'huggingface',
  'id': 25720743,
  'node_id': 'MDEyOk9yZ2FuaXphdGlvbjI1NzIwNzQz',
  'avatar_url': 'https://avatars.githubusercontent.com/u/25720743?v=4',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/huggingface',
  'html_url': 'https://github.com/huggingface',
  'followers_url': 'https://api.github.com/users/huggingface/followers',
  'following_url': 'https://api.github.com/users/huggingface/following{/other_user}',
  'gists_url': 'https://api.github.com/users/huggingface/gists{/gist_id}',
  'starred_url': 'https://api.github.com/users/huggingface/starred{/owner}{/repo}',
  'subscriptions_url': 'https://api.github.com/users/huggingface/subscriptions',
  'organizations_url': 'https://api.github.com/users/huggingface/orgs',
  'repos_url': 'https://api.github.com/users/huggingface/repos',
  'events_url': 

### Get issues

In [26]:
def get_issues(owner: str, repo: str, limit: int = 20):
    return github_get(
        f"/repos/{owner}/{repo}/issues",
        params={
            "state": "all",
            "per_page": limit,
        },
    )

In [28]:
issues = get_issues(
    owner,
    repo,
    config["github"]["limits"]["issues"],
)

len(issues)

100

In [31]:
# for issue in issues:
#     print(issue['number'])

In [30]:
issues[0]

{'url': 'https://api.github.com/repos/huggingface/transformers/issues/48266',
 'repository_url': 'https://api.github.com/repos/huggingface/transformers',
 'labels_url': 'https://api.github.com/repos/huggingface/transformers/issues/48266/labels{/name}',
 'comments_url': 'https://api.github.com/repos/huggingface/transformers/issues/48266/comments',
 'events_url': 'https://api.github.com/repos/huggingface/transformers/issues/48266/events',
 'html_url': 'https://github.com/huggingface/transformers/pull/48266',
 'id': 5241798706,
 'node_id': 'PR_kwDOCUB6oc8AAAABA4m1Ew',
 'number': 48266,
 'title': '[`GDN`] Fix recurrent FLA fallback',
 'user': {'login': 'vasqu',
  'id': 73884904,
  'node_id': 'MDQ6VXNlcjczODg0OTA0',
  'avatar_url': 'https://avatars.githubusercontent.com/u/73884904?v=4',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/vasqu',
  'html_url': 'https://github.com/vasqu',
  'followers_url': 'https://api.github.com/users/vasqu/followers',
  'following_url': 'https://ap

In [15]:
# filter them
issues = [issue for issue in issues if "pull_request" not in issue]

### Issue comments

In [16]:
def get_issue_comments(
    owner: str,
    repo: str,
    issue_number: int,
):
    return github_get(
        f"/repos/{owner}/{repo}/issues/{issue_number}/comments",
        params={
            "per_page": 100,
        },
    )

In [18]:
issue_comments = {}

for issue in issues:
    issue_number = issue["number"]

    issue_comments[issue_number] = get_issue_comments(
        owner,
        repo,
        issue_number,
    )

### Pull requests

In [32]:
def get_pull_requests(
    owner: str,
    repo: str,
    limit: int = 20,
):
    return github_get(
        f"/repos/{owner}/{repo}/pulls",
        params={
            "state": "all",
            "per_page": limit,
        },
    )

In [33]:
pull_requests = get_pull_requests(
    owner,
    repo,
    config["github"]["limits"]["pull_requests"],
)

len(pull_requests)

100

In [36]:
# for pull_request in pull_requests:
#     print(pull_request['number'])

In [34]:
pull_requests[0]

{'url': 'https://api.github.com/repos/huggingface/transformers/pulls/48266',
 'id': 4354323731,
 'node_id': 'PR_kwDOCUB6oc8AAAABA4m1Ew',
 'html_url': 'https://github.com/huggingface/transformers/pull/48266',
 'diff_url': 'https://github.com/huggingface/transformers/pull/48266.diff',
 'patch_url': 'https://github.com/huggingface/transformers/pull/48266.patch',
 'issue_url': 'https://api.github.com/repos/huggingface/transformers/issues/48266',
 'number': 48266,
 'state': 'open',
 'locked': False,
 'title': '[`GDN`] Fix recurrent FLA fallback',
 'user': {'login': 'vasqu',
  'id': 73884904,
  'node_id': 'MDQ6VXNlcjczODg0OTA0',
  'avatar_url': 'https://avatars.githubusercontent.com/u/73884904?v=4',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/vasqu',
  'html_url': 'https://github.com/vasqu',
  'followers_url': 'https://api.github.com/users/vasqu/followers',
  'following_url': 'https://api.github.com/users/vasqu/following{/other_user}',
  'gists_url': 'https://api.github.com/u

### PR reviews

In [39]:
def get_pr_reviews(
    owner: str,
    repo: str,
    pr_number: int,
):
    return github_get(
        f"/repos/{owner}/{repo}/pulls/{pr_number}/reviews",
        params={
            "per_page": 100,
        },
    )

In [40]:
pr_reviews = {}

for pr in pull_requests:
    pr_number = pr["number"]

    pr_reviews[pr_number] = get_pr_reviews(
        owner,
        repo,
        pr_number,
    )

In [45]:
pr_reviews[48266]

[{'id': 5014984492,
  'node_id': 'PRR_kwDOCUB6oc8AAAABKuqXLA',
  'user': {'login': 'ArthurZucker',
   'id': 48595927,
   'node_id': 'MDQ6VXNlcjQ4NTk1OTI3',
   'avatar_url': 'https://avatars.githubusercontent.com/u/48595927?u=0fb4420052bfff0133f96996c916ecde6ed6fa69&v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/ArthurZucker',
   'html_url': 'https://github.com/ArthurZucker',
   'followers_url': 'https://api.github.com/users/ArthurZucker/followers',
   'following_url': 'https://api.github.com/users/ArthurZucker/following{/other_user}',
   'gists_url': 'https://api.github.com/users/ArthurZucker/gists{/gist_id}',
   'starred_url': 'https://api.github.com/users/ArthurZucker/starred{/owner}{/repo}',
   'subscriptions_url': 'https://api.github.com/users/ArthurZucker/subscriptions',
   'organizations_url': 'https://api.github.com/users/ArthurZucker/orgs',
   'repos_url': 'https://api.github.com/users/ArthurZucker/repos',
   'events_url': 'https://api.github.com/users/Arth

### PR comments

In [46]:
def get_pr_comments(
    owner: str,
    repo: str,
    pr_number: int,
):
    return github_get(
        f"/repos/{owner}/{repo}/issues/{pr_number}/comments",
        params={
            "per_page": 100,
        },
    )

In [47]:
pr_comments = {}

for pr in pull_requests:
    pr_number = pr["number"]

    pr_comments[pr_number] = get_pr_comments(
        owner,
        repo,
        pr_number,
    )

In [49]:
pr_comments[48266]

[{'url': 'https://api.github.com/repos/huggingface/transformers/issues/comments/5404775437',
  'html_url': 'https://github.com/huggingface/transformers/pull/48266#issuecomment-5404775437',
  'issue_url': 'https://api.github.com/repos/huggingface/transformers/issues/48266',
  'id': 5404775437,
  'node_id': 'IC_kwDOCUB6oc8AAAABQiZUDQ',
  'user': {'login': 'github-actions[bot]',
   'id': 41898282,
   'node_id': 'MDM6Qm90NDE4OTgyODI=',
   'avatar_url': 'https://avatars.githubusercontent.com/in/15368?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/github-actions%5Bbot%5D',
   'html_url': 'https://github.com/apps/github-actions',
   'followers_url': 'https://api.github.com/users/github-actions%5Bbot%5D/followers',
   'following_url': 'https://api.github.com/users/github-actions%5Bbot%5D/following{/other_user}',
   'gists_url': 'https://api.github.com/users/github-actions%5Bbot%5D/gists{/gist_id}',
   'starred_url': 'https://api.github.com/users/github-actions%5Bbot%5D/sta

### Put everything together

In [50]:
data = {
    "repository": repository,
    "issues": issues,
    "issue_comments": issue_comments,
    "pull_requests": pull_requests,
    "pull_request_reviews": pr_reviews,
    "pull_request_comments": pr_comments,
}

In [53]:
output_dir = Path("data/raw/github")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / f"{owner}_{repo}.json"

with output_file.open("w") as f:
    json.dump(data, f, indent=2)

print(f"Saved: {output_file}")

Saved: data/raw/github/huggingface_transformers.json
